# Increasing Hidden Fraction (Hidden Units)

Grid search varying `hidden_unit_fraction` from 0.05 to 0.50. Neurons are fully removed from the student's weight matrix (not just unobserved). Training uses CMA-ES initialisation followed by gradient descent on scaling factors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import toml
import torch
import zarr

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.analysis.inference import run_feedforward_inference
from connectome_snns.configs.conductance_based import RecurrentLayerConfig, FeedforwardLayerConfig
from connectome_snns.analysis import (
    fluctuation_r_squared,
    interleave_spike_trains,
    make_grid_colormap,
    r_squared,
)
from connectome_snns.visualization import (
    SF_PATHWAYS,
    plot_firing_rate_scatter,
    plot_r2_vs_parameter,
    plot_spike_trains,
    use_project_style,
)

use_project_style()

In [ ]:
hidden_units_config = load_experiment_config("experiment.toml")
GRID_DIR = hidden_units_config["output_dir"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FRACTIONS = np.arange(0.05, 0.55, 0.05).round(2)
SMOOTH_WINDOW = 10
MAX_RATE = 40
TAU_MS = 50.0
N_NEURONS_SHOW = 10
T_SHOW_MS = 5000  # 5 seconds

# Load grid search metrics
grid_metrics = {}
for frac in FRACTIONS:
    csv_path = GRID_DIR / f"hidden-{frac:.2f}" / "training_metrics.csv"
    if csv_path.exists():
        grid_metrics[frac] = pd.read_csv(csv_path)
        print(f"hidden_unit_fraction={frac:.2f}: {len(grid_metrics[frac])} rows")

sorted_fracs = sorted(grid_metrics.keys())
norm, color_dict = make_grid_colormap(sorted_fracs)

In [ ]:
def compute_per_neuron_rates(exp_dir, device="cpu"):
    """Run student model inference and compute per-neuron firing rates.

    Uses run_feedforward_inference with output_indices to handle the
    visible-only subset. Results are cached to per_neuron_rates.npz.
    """
    cache_path = exp_dir / "per_neuron_rates.npz"
    if cache_path.exists():
        print(f"  Cached: {exp_dir.name}")
        return dict(np.load(cache_path))

    print(f"  Computing: {exp_dir.name}...")

    ckpt_path = exp_dir / "checkpoints" / "checkpoint_best.pt"
    if not ckpt_path.exists():
        ckpt_path = exp_dir / "checkpoints" / "checkpoint_latest.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    with open(exp_dir / "parameters.toml") as f:
        params = toml.load(f)
    rec_cfg = RecurrentLayerConfig(**params["recurrent"])
    ff_cfg = FeedforwardLayerConfig(**params["feedforward"])
    input_cell_type_names = ff_cfg.cell_types.names + rec_cfg.cell_types.names
    output_cell_type_names = rec_cfg.cell_types.names

    state = ckpt["model_state_dict"]
    n_in = len(input_cell_type_names)
    n_out = len(output_cell_type_names)
    learned_sf = np.ones((n_in, n_out), dtype=np.float32)
    for i, src in enumerate(input_cell_type_names):
        for j, tgt in enumerate(output_cell_type_names):
            key = f"ff_projections.{src}__{tgt}.log_sf"
            learned_sf[i, j] = float(torch.exp(state[key]).item())

    init = np.load(exp_dir / "initial_state" / "network_structure.npz")
    visible_indices = init["visible_indices"]
    hidden_indices = init["hidden_indices"]

    z = zarr.open_group(exp_dir / "inputs" / "spike_data.zarr", mode="r")
    chunk_size = params["simulation"]["chunk_size"]
    total_chunks = z["output_spikes"].shape[1] // chunk_size

    data = run_feedforward_inference(
        params_file=exp_dir / "parameters.toml",
        run_dir=exp_dir,
        scaling_factors_FF=learned_sf,
        output_indices=visible_indices,
        n_analysis_chunks=total_chunks - 25,
        cache_path=None,
        device=device,
        desc=exp_dir.name,
    )

    student_spikes = data["student_spikes"]
    target_spikes = data["teacher_spikes"]
    cell_type_indices = data["cell_type_indices"]
    dt_ms = float(data["dt"])

    duration_s = student_spikes.shape[0] * dt_ms / 1000.0
    student_rates = student_spikes.sum(axis=0) / duration_s
    teacher_rates_visible = target_spikes.sum(axis=0) / duration_s

    teacher_all = np.array(z["output_spikes"][0, :, :])
    teacher_rates_all = teacher_all.sum(axis=0) / (
        teacher_all.shape[0] * dt_ms / 1000.0
    )

    ns = np.load(exp_dir / "inputs" / "network_structure.npz")
    full_cell_type_indices = ns["cell_type_indices"]

    result = dict(
        student_rates=student_rates,
        teacher_rates_visible=teacher_rates_visible,
        teacher_rates_all=teacher_rates_all,
        student_spikes=student_spikes,
        target_spikes=target_spikes,
        visible_indices=visible_indices,
        hidden_indices=hidden_indices,
        cell_type_indices=cell_type_indices,
        full_cell_type_indices=full_cell_type_indices,
        dt=dt_ms,
    )
    np.savez(cache_path, **result)

    if device == "cuda":
        torch.cuda.empty_cache()

    print(f"    Done: {len(visible_indices)} visible, {len(hidden_indices)} hidden")
    return result

## CMA-ES Initialisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: best loss over evaluations
ax = axes[0]
for frac, df in sorted(grid_metrics.items()):
    cma = df[df["epoch"] < 0].copy()
    if cma.empty:
        continue
    ax.plot(
        cma["cma_es/n_evals"],
        cma["cma_es/best_loss"],
        color=color_dict[frac],
        linewidth=1.2,
        label=f"{frac:.2f}",
    )
ax.set_xlabel("CMA-ES Evaluations")
ax.set_ylabel("Best Loss")
ax.set_title("CMA-ES Convergence")
ax.legend(title="Hidden Frac.", fontsize=7, ncol=2)

# Right: sigma over evaluations
ax = axes[1]
for frac, df in sorted(grid_metrics.items()):
    cma = df[df["epoch"] < 0].copy()
    if cma.empty:
        continue
    ax.plot(
        cma["cma_es/n_evals"],
        cma["cma_es/sigma"],
        color=color_dict[frac],
        linewidth=1.2,
    )
ax.set_xlabel("CMA-ES Evaluations")
ax.set_ylabel("Sigma")
ax.set_title("CMA-ES Step Size")

plt.suptitle("CMA-ES Initialisation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Scaling Factor Convergence

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True)

for idx, (key, title) in enumerate(SF_PATHWAYS):
    ax = axes.flat[idx]
    col = f"scaling_factors/{key}_value"
    target_col = f"scaling_factors/{key}_target"

    for frac, df in sorted(grid_metrics.items()):
        # Gradient phase only (positive epochs)
        grad = df[df["epoch"] > 0]
        if col not in grad.columns:
            continue
        ax.plot(
            grad["epoch"],
            grad[col],
            color=color_dict[frac],
            linewidth=1.2,
            label=f"{frac:.2f}" if idx == 0 else None,
        )

    ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Scaling Factor")
    if idx >= 4:
        ax.set_xlabel("Epoch")
    if idx == 0:
        ax.legend(title="Hidden Frac.", fontsize=7, loc="best", ncol=2)

plt.suptitle(
    "Scaling Factor Convergence (Gradient Phase)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## Final Scaling Factors vs Hidden Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: individual scaling factors
ax = axes[0]
for key, title in SF_PATHWAYS:
    col = f"scaling_factors/{key}_value"
    values = [grid_metrics[f].iloc[-1][col] for f in sorted_fracs]
    ax.plot(sorted_fracs, values, "o-", label=title, markersize=5)

ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Hidden Unit Fraction")
ax.set_ylabel("Final Scaling Factor")
ax.set_title("Individual Scaling Factors")
ax.legend(fontsize=7)

# Right: mean absolute deviation from target
ax = axes[1]
mean_devs = []
for frac in sorted_fracs:
    final = grid_metrics[frac].iloc[-1]
    sf_vals = np.array([final[f"scaling_factors/{k}_value"] for k, _ in SF_PATHWAYS])
    sf_tgts = np.array([final[f"scaling_factors/{k}_target"] for k, _ in SF_PATHWAYS])
    mean_devs.append(np.mean(np.abs(sf_vals - sf_tgts)))

ax.plot(sorted_fracs, mean_devs, "ko-", markersize=6)
ax.set_xlabel("Hidden Unit Fraction")
ax.set_ylabel("Mean |SF - target|")
ax.set_title("Mean Scaling Factor Error")

plt.suptitle(
    "Final Scaling Factors vs Hidden Unit Fraction", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Save left panel only
fig_left, ax_left = plt.subplots(figsize=(6, 4))
for key, title in SF_PATHWAYS:
    col = f"scaling_factors/{key}_value"
    values = [grid_metrics[f].iloc[-1][col] for f in sorted_fracs]
    ax_left.plot(sorted_fracs, values, "o-", label=title, markersize=5)

ax_left.axhline(y=1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax_left.set_xlabel("Hidden Unit Fraction")
ax_left.set_ylabel("Final Scaling Factor")
ax_left.set_title("Final Scaling Factors vs Hidden Unit Fraction", fontweight="bold")
ax_left.legend(fontsize=7)
fig_left.tight_layout()
plt.show()

## Loss vs Hidden Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss trajectories (gradient phase)
ax = axes[0]
for frac, df in sorted(grid_metrics.items()):
    grad = df[df["epoch"] > 0]
    smoothed = grad["van_rossum_loss"].rolling(SMOOTH_WINDOW).mean()
    ax.plot(
        grad["epoch"],
        smoothed,
        color=color_dict[frac],
        linewidth=1.2,
        label=f"{frac:.2f}",
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Van Rossum Loss")
ax.set_title("Loss Trajectories (Gradient Phase)")
ax.set_ylim(0, None)
ax.legend(title="Hidden Frac.", fontsize=7, ncol=2)

# Right: final loss vs fraction
ax = axes[1]
final_losses = [grid_metrics[f].iloc[-1]["van_rossum_loss"] for f in sorted_fracs]
ax.plot(sorted_fracs, final_losses, "ko-", markersize=6)
ax.set_xlabel("Hidden Unit Fraction")
ax.set_ylabel("Final Van Rossum Loss")
ax.set_title("Final Loss")

plt.suptitle("Loss vs Hidden Unit Fraction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Firing Rates vs Hidden Fraction

In [ ]:
fr_grid = [
    (0, 0, "excitatory", "Excitatory"),
    (0, 1, "inhibitory", "Inhibitory"),
]

for stat, stat_label, suptitle in [
    ("mean", "Mean Firing Rate (Hz)", "Mean Firing Rates vs Hidden Unit Fraction"),
    ("std", "Std Firing Rate (Hz)", "Firing Rate Std vs Hidden Unit Fraction"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for col_idx, (_, _, cell_type, title) in enumerate(fr_grid):
        ax = axes[col_idx]
        student_col = f"firing_rate/student_{cell_type}_{stat}"
        teacher_col = f"firing_rate/teacher_{cell_type}_{stat}"

        for frac, df in sorted(grid_metrics.items()):
            grad = df[df["epoch"] > 0]
            if student_col not in grad.columns:
                continue
            x = grad["epoch"]
            y = grad[student_col].rolling(SMOOTH_WINDOW).mean()
            ax.plot(
                x,
                y,
                color=color_dict[frac],
                linewidth=1.2,
                label=f"{frac:.2f}" if col_idx == 0 else None,
            )

        first_df = next(iter(grid_metrics.values()))
        grad_first = first_df[first_df["epoch"] > 0]
        if teacher_col in grad_first.columns:
            t_y = grad_first[teacher_col].rolling(SMOOTH_WINDOW).mean()
            ax.plot(
                grad_first["epoch"],
                t_y,
                color="black",
                linestyle="--",
                linewidth=1.5,
                label="Teacher" if col_idx == 0 else None,
            )

        ax.set_title(title)
        ax.set_ylim(0, None)
        ax.set_xlabel("Epoch")
        if col_idx == 0:
            ax.set_ylabel(stat_label)
            ax.legend(title="Hidden Frac.", fontsize=7, ncol=2)

    plt.suptitle(suptitle, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Summary

## Per-Neuron Firing Rates (Inference with Final Scaling Factors)

In [ ]:
# Compute all rates
all_rates = {}
for frac in sorted_fracs:
    exp_dir = GRID_DIR / f"hidden-{frac:.2f}"
    all_rates[frac] = compute_per_neuron_rates(exp_dir, DEVICE)

for frac in sorted_fracs:
    rates = all_rates[frac]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    plot_firing_rate_scatter(
        rates["teacher_rates_visible"],
        rates["student_rates"],
        rates["cell_type_indices"],
        split_panels=True,
        axes=[axes[0], axes[1]],
        max_rate=MAX_RATE,
        marker_size=4,
        alpha=0.3,
    )

    fig.suptitle(
        f"Hidden Fraction = {frac:.2f}",
        fontsize=14,
        fontweight="bold",
    )
    fig.tight_layout()
    if frac == 0.5:
        fig.savefig(
            GRID_DIR / f"firing_rate_scatter_{frac:.2f}.svg", bbox_inches="tight"
        )
        plt.show()
    else:
        plt.close(fig)

## R² vs Hidden Fraction

In [ ]:
r2_vals = []
for frac in sorted_fracs:
    rates = all_rates[frac]
    r2_vals.append(r_squared(rates["teacher_rates_visible"], rates["student_rates"]))

plot_r2_vs_parameter(
    sorted_fracs,
    {"Visible": r2_vals},
    xlabel="Hidden Unit Fraction",
    title="Per-Neuron Firing Rate R\u00b2 vs Hidden Fraction",
)
plt.show()

## Spike Train Fluctuation R² vs Hidden Fraction

Gaussian-smoothed (σ = 50 ms) spike trains compared between teacher and student for visible neurons. Measures how well temporal fluctuations are reproduced, not just mean rates.

In [ ]:
r2_fluct_visible = []

for frac in sorted_fracs:
    rates = all_rates[frac]
    exp_dir = GRID_DIR / f"hidden-{frac:.2f}"
    cache_path = exp_dir / "fluct_r2.npz"

    r2_vis = None
    if cache_path.exists():
        cached = np.load(cache_path)
        if "r2_visible" in cached:
            r2_vis = float(cached["r2_visible"])
            print(f"frac={frac:.2f}  R\u00b2(visible)={r2_vis:.4f}  (cached)")

    if r2_vis is None:
        dt_ms = float(rates["dt"])
        teacher_sp = rates["target_spikes"].astype(np.float32)
        student_sp = rates["student_spikes"].astype(np.float32)

        r2_vis = fluctuation_r_squared(teacher_sp, student_sp, TAU_MS, dt_ms)

        np.savez(cache_path, r2_visible=r2_vis)
        print(f"frac={frac:.2f}  R\u00b2(visible)={r2_vis:.4f}")

    r2_fluct_visible.append(r2_vis)

plot_r2_vs_parameter(
    sorted_fracs,
    {"Visible": r2_fluct_visible},
    xlabel="Hidden Unit Fraction",
    title=f"Spike Train Fluctuation R\u00b2 vs Hidden Fraction  (\u03c3 = {TAU_MS:.0f} ms)",
)
plt.show()

## Spike Raster Comparisons

Teacher vs student spike trains for visible neurons, plotted for each hidden fraction.

In [ ]:
all_visible_sets = [set(all_rates[f]["visible_indices"]) for f in sorted_fracs]
always_visible = np.array(sorted(set.intersection(*all_visible_sets)))
print(f"Always-visible neurons: {len(always_visible)}")

rng = np.random.RandomState(42)
show_neuron_ids = np.sort(rng.choice(always_visible, N_NEURONS_SHOW, replace=False))

for frac in sorted_fracs:
    rates = all_rates[frac]
    dt_ms = float(rates["dt"])
    vis_idx = rates["visible_indices"]

    vis_id_to_col = {int(nid): col for col, nid in enumerate(vis_idx)}
    cols = [vis_id_to_col[nid] for nid in show_neuron_ids]

    t_end = min(int(T_SHOW_MS / dt_ms), rates["student_spikes"].shape[0])
    teacher = rates["target_spikes"][:t_end, cols]
    student = rates["student_spikes"][:t_end, cols]

    interleaved, ct_idx = interleave_spike_trains(
        teacher, student, n_neurons=N_NEURONS_SHOW
    )

    fig = plot_spike_trains(
        spikes=interleaved,
        dt=dt_ms,
        cell_type_indices=ct_idx,
        cell_type_names=["Teacher", "Student"],
        n_neurons_plot=2 * N_NEURONS_SHOW,
        n_compared=2,
        fraction=1.0,
        random_seed=None,
        title=f"Visible Neurons \u2014 Hidden Fraction = {frac:.2f}",
        figsize=(16, 6),
    )
    if frac == 0.5:
        plt.show()

In [ ]:
rows = []
for frac in sorted_fracs:
    final = grid_metrics[frac].iloc[-1]
    sf_vals = np.array([final[f"scaling_factors/{k}_value"] for k, _ in SF_PATHWAYS])
    sf_tgts = np.array([final[f"scaling_factors/{k}_target"] for k, _ in SF_PATHWAYS])

    rows.append(
        {
            "Hidden Frac.": f"{frac:.2f}",
            "Final Loss": f"{final['van_rossum_loss']:.2f}",
            "FR Exc (Hz)": f"{final['firing_rate/student_excitatory_mean']:.2f}",
            "FR Inh (Hz)": f"{final['firing_rate/student_inhibitory_mean']:.2f}",
            "SF Range": f"[{sf_vals.min():.3f}, {sf_vals.max():.3f}]",
            "Mean |SF - target|": f"{np.mean(np.abs(sf_vals - sf_tgts)):.3f}",
        }
    )

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))